# Partial Replication of Anthropic's Emotion Vectors with *Gemma 4 E2B* and *GPT 2 Medium* inside Google Colab T4 Notebook

Abraham Jhared Flores Azcona

abrahamjhared.flores@gmail.com

April 25, 2026


## Environment Dependencies
**[IMPORTANT]** `transformers==5.5.0` is required to access Gemma 4 E2B from HuggingFace.

**[IMPORTANT]** `kaleido==0.2.1` is required to download plots generated from this Colab AND to remove confusing *"package not found"* error when using recent version.

In [ ]:
# Core Machine Learning & TPU Support
%pip install torch torch_xla[tpu] -f https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.1-cp310-cp310-linux_x86_64.whl
%pip install transformers==5.5.0 accelerate

# Interpretability & Visualization
%pip install plotly pandas scikit-learn huggingface-hub
%pip install -U kaleido==0.2.1

## Python snippets

### [1] Load relevant pip packages & global variables

**Description of global variables**

*   `kOutDir` path for output directory of the Colab (in this case, `./research_data`).
*   `gAccelerator` type of hardware accelerator used in the Colab.
*   `gDevice` type of hardware component; CPU or GPU.
*   `gTokenizer` instance of LLM tokenizer.
*   `gTargetLayer` number of the desired layer where "emotion" vector semantics arise.
*   `gStoryFile` path to `emotion_stories`.
*   `gEmotionLibrary` dictionary containing the name of the emotion (i.e: happy, calm, etc.), and the vector representation stored as *bfloat16*.
*   `gNeutralVectors` list containing the neutral vectors stored as *bfloat16*.

In [3]:
import time
import json
import os
import gc
import zipfile
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import plotly.express as px
from typing import List, Dict
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from google.colab import files


# Global variables for the Collab refactor
kOutDir = "./research_data"
gAccelerator = None
gDevice = None
gTokenizer = None
gModel = None
gTargetLayer = None # Layer 24 has consistent emotion classifications
gStoryFile = None
gEmotionLibrary: Dict[str, torch.Tensor] = None
gNeutralVectors: List[torch.Tensor] = None

### [2] Load desired model

**Description of global variables**

* `kModelIdx` HuggingFace's ID for desired LLM model hosted in their platform.

In [4]:
kModelIdx = "openai-community/gpt2-medium"

In [ ]:
kModelIdx = "google/gemma-4-E2B"

### [3] Load relevant "emotion" vector data

**Description of global variables**

* `neutralPrompts` list of 50 entries depicting *neutral* prompts, which consist of statements and orders.
* `emotionLabels` list of desired emotions for vector extraction.
* `storyTopics` list of ten topics for emotion story generation.


In [5]:
# @title
# Neutral prompt list from rain1955/emotion-vector-replication extendend with Google Gemini 3 Fast (04/09/2026)
neutralPrompts = [
  # prompts originating from rain1955/emotion-vector-replication
  "The meeting is scheduled for 3pm tomorrow.",
  "Please find the attached document.",
  "The temperature today is 22 degrees Celsius.",
  "The project deadline has been moved to next Friday.",
  "The store is located on the corner of Main Street.",
  "Chapter 3 discusses the economic implications.",
  "The software update includes several bug fixes.",
  "The report contains data from the past quarter.",
  "The committee will review the proposal next week.",
  "The library opens at 9am on weekdays.",
  # prompts generated with Google Gemini 3 Fast (04/09/2026)
  "The itinerary for the conference has been finalized.",
  "Data collection will commence at the beginning of the month.",
  "The user manual provides instructions for hardware setup.",
  "Standard procedure requires a signed authorization form.",
  "The server maintenance is performed every Sunday night.",
  "Historical records are stored in the basement archive.",
  "The chemical reaction occurs at room temperature.",
  "Please submit your expenses by the end of the day.",
  "The laboratory results will be available in 48 hours.",
  "The publication follows a strict peer-review process.",
  "The office is situated on the fourth floor of the building.",
  "Annual inspections are mandatory for all equipment.",
  "The software license expires at the end of the calendar year.",
  "The lecture series covers fundamental principles of physics.",
  "The supply chain manager coordinates all logistics.",
  "The router configuration remains unchanged since the last update.",
  "Participant feedback is collected through an anonymous survey.",
  "The contract specifies the terms and conditions of employment.",
  "Geological surveys indicate a high concentration of minerals.",
  "The system generates a log file for every transaction.",
  "Traffic flow is monitored by automated sensors.",
  "The workshop focuses on improving technical documentation.",
  "The final audit confirmed the accuracy of the financial statements.",
  "The museum is closed for renovations until next month.",
  "Utility bills are calculated based on monthly consumption.",
  "The inventory count is updated every Tuesday morning.",
  "The parking garage remains open 24 hours a day.",
  "The application process requires a valid form of identification.",
  "The thermostat is set to maintain a constant temperature.",
  "The user interface supports three different language options.",
  "The flight departure is confirmed for gate 12B.",
  "The building specifications meet the current safety codes.",
  "The printer requires a replacement toner cartridge.",
  "The database backup was completed successfully at midnight.",
  "The street lights are programmed to activate at sunset.",
  "The employee handbook outlines the company's privacy policy.",
  "The water filtration system is inspected twice a year.",
  "The package dimensions must not exceed the standard limit.",
  "The conference call will be recorded for future reference.",
  "The technical support team is available via email."
]

In [6]:
# @title
# Emotion word list from rain1955/emotion-vector-replication
emotionLabels = [
    'calm', 'loving', 'sad', 'guilty', 'desperate', 'afraid', 'angry', 'surprised', 'happy'
]

# Topics from rain1955/emotion-vector-replication
storyTopics = [
    "a student preparing for an exam",
    "a chef cooking a meal for guests",
    "a parent watching their child play",
    "a soldier returning home",
    "an artist finishing a painting",
    "a driver stuck in traffic",
    "a doctor delivering news to a patient",
    "a traveler arriving in a new city",
    "a musician performing on stage",
    "a shopkeeper closing for the day",
]

If you wish, you can load the 20-emotions variant of `emotionLabels`. Plots and Emotion Vector Generation will vary.

In [7]:
# Emotion word list from rain1955/emotion-vector-replication
emotionLabels = [
  "happy", "sad", "angry", "afraid", "calm",
  "desperate", "loving", "guilty", "surprised", "nervous",
  "proud", "inspired", "spiteful", "brooding", "playful",
  "anxious", "confused", "disgusted", "lonely", "hopeful"
]

### [4] Load relevant helper functions

**Description of `initialize()` function**

* Initializes previously-described global variables of the Colab environment.

**Description of `freeVRAM()` function**

* If using an NVIDIA GPU, clears and frees cached VRAM. Useful to minimize VRAM footprint, which spikes during emotion vector usage.

**Description of `normalizeVector()` function**

* *@input:* `vector` a PyTorch vector.
* Helper function wrapper to normalize a vector to unit length.

**Description of `computeCosineSimilarity()` function**

* *@input:* `vectorA` a PyTorch vector.
* *@input:* `vectorB` a PyTorch vector.
* Helper function wrapper to calculate cosine similarity.

In [8]:
def initialize():
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    print(f"[INIT] Initializing Research Orchestrator for {kModelIdx}...")
    gAccelerator = Accelerator()
    gDevice = gAccelerator.device
    gTokenizer = AutoTokenizer.from_pretrained(kModelIdx)
    gTokenizer.padding_side = "right" # gpt 2 setting
    if gTokenizer.pad_token is None:
        gTokenizer.pad_token = gTokenizer.eos_token
    gModel = AutoModelForCausalLM.from_pretrained(
        kModelIdx,
        torch_dtype=torch.bfloat16
    ).to(gDevice)
    gModel.eval()
    gEmotionLibrary = {}
    gNeutralVectors = []
    gTargetLayer = 0
    gStoryFile = os.path.join(kOutDir, "emotion_stories.json")
    print(f"[INIT] Model loaded. Target Layer: {gTargetLayer} | Device: {gDevice}")

def freeVRAM():
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gAccelerator.free_memory()

def normalizeVector(vector):
    vector = vector.view(-1)  # force 1D
    return vector / (vector.norm() + 1e-9)

def computeCosineSimilarity(vectorA, vectorB):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    return F.cosine_similarity(vectorA.unsqueeze(0), vectorB.unsqueeze(0)).item()

**Description of `getExistingKeys()` function**

* Helper function wrapper unique JSON (emotion, topic, sample) tuples on disk.

In [9]:
def getExistingKeys() -> set:
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Checkpointing: Identifies unique (emotion, topic, sample) tuples on disk."""
    existingKeys = set()
    if os.path.exists(gStoryFile):
        with open(gStoryFile, "r", encoding="utf-8") as fileHandle:
            for line in fileHandle:
                try:
                    entryData = json.loads(line)
                    existingKeys.add(f"{entryData['emotion']}_{entryData['topic_idx']}_{entryData['story_idx']}")
                except: continue
    return existingKeys

**Description of `generateVignettes()` function**

* *@input:* `promptInput` text prompt to generate a story from.
* *@input:* `nSamples` number of samples to generate. *Sets to 1 by default.*
* Helper function wrapper to normalize a vector to unit length.

In [10]:
def generateVignettes(promptInput: str, nSamples: int = 1, category: str = "Unset") -> List[str]:
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    gTokenizer.padding_side = "left"
    tokenizedInputs = gTokenizer(promptInput, padding=True, return_tensors="pt").to(gDevice)
    inputLength = tokenizedInputs['input_ids'].shape[1]
    vignetteList = []
    for _ in range(nSamples):
        outputTokens = gModel.generate(
            **tokenizedInputs, max_new_tokens=150, temperature=0.85, do_sample=True,
            pad_token_id=gTokenizer.pad_token_id, eos_token_id=gTokenizer.eos_token_id
        )
        vignetteList.append(gTokenizer.decode(outputTokens[0][inputLength:], skip_special_tokens=True).strip())
    return vignetteList

**Description of `generateStructuredStories()` function**

* *@input:* `emotions` list of emotions to generate expected stories from.
* *@input:* `topics` list of topics to based our story content from.
* *@input:* `samplesPerPair` number of samples to generate from (emotion, topic).
* Generates desired stories with the list of emotions and topics, and stores them into disk with a JSON file.

In [11]:
def generateStructuredStories(emotions: List[str], topics: List[str], samplesPerPair: int = 5):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Generates the grounded vignette dataset for vector extraction."""
    existingKeys = getExistingKeys()
    with open(gStoryFile, "a", encoding="utf-8") as fileHandle:
        for emotionIndex, emotionLabel in enumerate(emotions):
            for topicIndex, topicText in enumerate(topics):
                for sampleIndex in range(samplesPerPair):
                    uniqueKey = f"{emotionLabel}_{topicIndex}_{sampleIndex}"
                    if uniqueKey in existingKeys: continue

                    promptContent = f"Write a short paragraph about {topicText}. The character is feeling {emotionLabel}. Output only the paragraph."
                    chatMessages = [{"role": "user", "content": promptContent}]
                    formattedPrompt = gTokenizer.apply_chat_template(chatMessages, tokenize=False, add_generation_prompt=True)

                    generatedStory = generateVignettes(formattedPrompt, nSamples=1, category=f"{emotionLabel}/{topicText[:10]}")[0]
                    storyRecord = {
                        "emotion": emotionLabel, "topic_idx": topicIndex, "topic": topicText,
                        "story_idx": sampleIndex, "text": generatedStory, "timestamp": time.time()
                    }
                    fileHandle.write(json.dumps(storyRecord, ensure_ascii=False) + "\n")
                    fileHandle.flush()
                    existingKeys.add(uniqueKey)
            freeVRAM()

**Description of `getHiddenRepresentation()` function**

* *@input:* `promptList` list of text prompts to be used for hidden rerpesentation (activation) extraction.
* *@input:* `layerIndex` number representing the layer where activation extraction must happen.
* *@input:* `lastNTokens` number of last tokens where we'll extract activation from.
* Obtains the hidden state representation of the text prompt at a given layer.

**Description of `getBatchActivations()` function**

* *@input:* `textList` list of text prompts to be used for hidden representation (activation) extraction.
* *@input:* `layerIndex` number representing the layer where activation extraction must happen.
* Function wrapper of `getHiddenRepresentation()`.

In [12]:
def getHiddenRepresentation(promptList: List[str], layerIndex: int, lastNTokens: int = 50) -> torch.Tensor:
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile

    gTokenizer.padding_side = "right" # gpt 2 setting
    tokenizedBatch = gTokenizer(promptList, return_tensors="pt", truncation=True, padding=True).to(gDevice)

    with torch.no_grad():
        outputs = gModel(**tokenizedBatch, output_hidden_states=True)

    hiddenStates = outputs.hidden_states[layerIndex]  # [B, T, D]
    attentionMask = tokenizedBatch["attention_mask"]  # [B, T]

    batchVectors = []
    for i in range(hiddenStates.shape[0]):
        seqLen = int(attentionMask[i].sum().item())
        startIdx = max(0, seqLen - lastNTokens)
        vector = hiddenStates[i, startIdx:seqLen, :].mean(dim=0)
        batchVectors.append(vector)

    return torch.stack(batchVectors)

def captureBatchActivations(textList: List[str], layerIndex: int) -> torch.Tensor:
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    return getHiddenRepresentation(textList, layerIndex)

**Description of `extractEmotionVector()` function**

* *@input:* `emotionLabel` name of the desired emotion to extract its vector representation.
* *@input:* `neutralTexts` list of statements and orders used as noise.
* Extracts the emotion stories from a JSON file, captures the hidden activations representative of the stories, and saves the vectors into our runtime emotion library.

In [13]:
# Redefine extractEmotionVector with batching and JSONL parsing fix
def extractEmotionVector(emotionLabel: str, neutralTexts: List[str]):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile

    print(f"[EXTRACT] Emotion: {emotionLabel.upper()} | Layer: {gTargetLayer}")

    emotionalTexts = []

    # Correct variable name: emotion -> emotionLabel
    filePath = os.path.join(kOutDir, f"emotion_stories/{emotionLabel}_stories.json")
    if os.path.exists(filePath):
        with open(filePath, "r") as f:
            # Correct JSON loading for JSONL format
            dataList = json.load(f) # Note: json.load(), not loads()
            for d in dataList:
                emotionalTexts.append(d['text'])

    if not emotionalTexts:
        print(f"[WARN] No emotional texts found for {emotionLabel}. Skipping.")
        return None

    # Introduce batching for processing emotionalTexts before calling captureBatchActivations
    batchSize = 8 # Adjusted batch size for GPU memory. This can be tuned.
    allActivations = []

    for i in range(0, len(emotionalTexts), batchSize):
        batchEmotionalTexts = emotionalTexts[i:i + batchSize]
        if batchEmotionalTexts: # Ensure batch is not empty
            activationsBatch = captureBatchActivations(batchEmotionalTexts, gTargetLayer)
            allActivations.append(activationsBatch)
            # It's good practice to free memory explicitly when dealing with OOM
            del activationsBatch
            torch.cuda.empty_cache()

    if not allActivations:
        print(f"[WARN] No activations were generated for {emotionLabel}. Skipping.")
        return None

    # Concatenate all batched activations
    activationsVector = torch.cat(allActivations, dim=0)

    # Store RAW mean (baseline subtraction later)
    rawMeanVector = activationsVector.mean(dim=0).float()
    gEmotionLibrary[emotionLabel] = rawMeanVector

    return None

**Description of `normalizeEmotionVectors()` function**


* Normalizes emotion vectors and neutral vectors, and updates emotion vectors by subtracting the global mean of the emotions and the neutral.

In [14]:
def normalizeEmotionVectors():
    global gEmotionLibrary, gNeutralVectors, gDevice

    if gNeutralVectors is None or len(gNeutralVectors) == 0:
        raise ValueError("Neutral vectors must be computed before finalization.")

    # --- STEP 1: Neutral baseline ---
    neutralMean = gNeutralVectors.mean(dim=0)

    for emotionKey in gEmotionLibrary:
        gEmotionLibrary[emotionKey] = gEmotionLibrary[emotionKey] - neutralMean

    # --- STEP 2: Global emotion centering (CRITICAL) ---
    stacked = torch.stack(list(gEmotionLibrary.values()))
    globalMean = stacked.mean(dim=0)

    for emotionKey in gEmotionLibrary:
        gEmotionLibrary[emotionKey] = gEmotionLibrary[emotionKey] - globalMean

    # --- STEP 3: Normalize ONLY ONCE at the end ---
    for emotionKey in gEmotionLibrary:
        vector = gEmotionLibrary[emotionKey]
        vector = vector / (vector.norm() + 1e-9)
        gEmotionLibrary[emotionKey] = vector.to(torch.bfloat16).to(gDevice)

    print("[FINALIZE] Emotion vectors centered + normalized.")

**Description of `extractNeutralVectors()` function**


* Special wrapper of `extractEmotionVectors()` for neutral prompts.

In [15]:
def extractNeutralVectors(neutralTexts: List[str]):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    print(f"[EXTRACT] Neutral | Layer: {gTargetLayer}")
    gNeutralVectors = captureBatchActivations(neutralTexts, gTargetLayer)

**Description of `printEmotionLogits()` function**

* *@input:* `emotionLabel` name of the desired emotion to print its top logit tokens
* *@input:* `topK` number of clusters to extract logits from. *Defaults to 10*.
* Prints the most relevant tokens related to an emotion vector. We standardize the logit lens to extract them from a Bell Curve distribution.

In [16]:
# @title
def printEmotionLogits(emotionLabel: str, topK: int = 10):
    global gModel, gTokenizer, gEmotionLibrary

    # [1] Prepare vector and precision
    emotionVector = gEmotionLibrary[emotionLabel].to(gModel.device).to(gModel.dtype)

    # [2] ROBUST FIX FOR GEMMA 4 E2B / MODERN LLAMA
    # Apply final LayerNorm (The 'secret sauce' for orientation)
    with torch.no_grad():
        if hasattr(gModel, "model"):
            if hasattr(gModel.model, "norm"):
                emotionVector = gModel.model.norm(emotionVector)
            elif hasattr(gModel.model, "final_layernorm"):
                emotionVector = gModel.model.final_layernorm(emotionVector)
        elif hasattr(gModel, "transformer") and hasattr(gModel.transformer, "ln_f"):
            emotionVector = gModel.transformer.ln_f(emotionVector)

        # [3] Pass through LM Head
        # Note: unsqueeze(0) and squeeze(0) handle the expected batch dimension
        logits = gModel.lm_head(emotionVector.unsqueeze(0)).squeeze(0)

        # [4] Z-SCORE STANDARDIZATION (Fixes nonsense percentages)
        # We calculate deviation from the vocabulary mean
        logitMean = logits.mean()
        logitStdDev = logits.std()
        zScores = (logits - logitMean) / logitStdDev

    # [5] Retrieve Top-K based on Z-Scores
    topValues, topIndices = torch.topk(zScores, topK)

    print(f"\n[LOGIT LENS] Semantic Strength for '{emotionLabel.upper()}':")
    for i in range(topK):
        # Decode and format output
        token = gTokenizer.decode([topIndices[i].item()])
        sigmaScore = topValues[i].item()

        # We print +N.NNσ to show standard deviations from the mean
        print(f"{i+1}. {token.strip():<15} (+{sigmaScore:.2f}σ)")

    return None

**Description of `get_layer_module()` function**

* *@input:* `model` name of the desired emotion to print its top logit tokens
* *@input:* `target_idx` number of the desired target layer.
* Obtains the vector layer representation expected of Gemma 4 and GPT 2 models.

In [17]:
# --- PHASE 1: DYNAMIC LAYER IDENTIFICATION ---
def get_layer_module(model, target_idx):
    # Path A: Standard Gemma/Llama nesting
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers[target_idx]

    # Path B: Gemma 4 E2B conditional generation wrapper
    if hasattr(model, "language_model"):
        return get_layer_module(model.language_model, target_idx)

    # Path C: Brute-force search for the primary ModuleList
    # This is the most robust fallback for non-standard E2B implementations
    for _, module in model.named_modules():
        if isinstance(module, torch.nn.ModuleList) and len(module) > target_idx:
            return module[target_idx]

    return None

**Description of `performSingularEmotionProbeSteering()` function**

* *@input:* `emotionVector` PyTorch vector of the desired emotion.
* *@input:* `inputPrompt` text prompt that must be provided to the model.
* *@input:* `steeringValue` scalar value for steering strength.
* Generates a text prompt altered by a singular emotion vector scaled by steering strength.

In [18]:
def performSingularEmotionProbeSteering(emotionVector, inputPrompt, steeringValue):
    global gModel, gTokenizer, gTargetLayer, gDevice

    # Ensure vector is on the right device and dtype
    # Gemma 4 often uses bfloat16; match the model's dtype
    emotionVector = emotionVector.to(gDevice).to(gModel.dtype)

    def steeringHook(module, input, output):
        # Handle the standard (hidden_states, optional_tuple) output
        if isinstance(output, tuple):
            hiddenStates = output[0]
        else:
            hiddenStates = output

        # Scale relative to the current activation magnitude (standardized steering)
        # We use a small epsilon to avoid division by zero
        scale = hiddenStates.norm(dim=-1, keepdim=True)

        # Prepare the steering delta
        # Shape must be [Batch, SeqLen, HiddenDim]
        steeringDelta = (steeringValue * scale * emotionVector)

        # Apply steering to EVERY token in the current pass
        # During 'generate', after the first token, seq_len is usually 1,
        # so this naturally targets the "current" token being predicted.
        steeredStates = hiddenStates + steeringDelta

        if isinstance(output, tuple):
            return (steeredStates,) + output[1:]
        return steeredStates

    # ROBUST ARCHITECTURE CHECK (Supports GPT2 and Gemma 4)
    targetLayer = get_layer_module(gModel, gTargetLayer)
    hookHandle = targetLayer.register_forward_hook(steeringHook)

    try:
        inputTokens = gTokenizer(inputPrompt, return_tensors="pt").to(gDevice)

        # Use a slightly lower temperature for Gemma 4 to see the steering effect clearly
        outputTokens = gModel.generate(
            **inputTokens,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=gTokenizer.eos_token_id # Gemma often uses EOS as PAD
        )
    finally:
        hookHandle.remove()

    outputText = gTokenizer.decode(outputTokens[0], skip_special_tokens=True)
    #print(f"\n[STEERING] Value: {steeringValue} | Prompt: {inputPrompt[:50]}...")
    #print(f"Output:\n{outputText}")
    return outputText

**Description of `superviseSingularEmotionProbeActivation()` function**

* *@input:* `emotionVector` PyTorch vector of the desired emotion.
* *@input:* `inputPrompt` text prompt that must be provided to the model.
* Computes the cosine similarity score of the emotion vector and the provided text prompt.

In [19]:
def superviseSingularEmotionProbeActivation(emotionVector, inputPrompt):
    """
    Identifies the model's internal layers dynamically and supervises
    the alignment between latent activations and target emotion vectors.
    """
    global gModel, gTokenizer, gTargetLayer, gDevice

    activationBuffer = []

    def observationHook(module, input, output):
        # Handle the Gemma 4 (hidden_states, cache) tuple output
        hiddenStates = output[0] if isinstance(output, tuple) else output
        # Move to CPU immediately to prevent GPU memory saturation
        activationBuffer.append(hiddenStates.detach().cpu())
        return output

    vectorLayer = get_layer_module(gModel, gTargetLayer)

    if vectorLayer is None:
        raise ValueError(f"CRITICAL: Failed to locate Layer {gTargetLayer}. "
                         f"Check model structure: {type(gModel)}")

    # --- PHASE 2: HOOK REGISTRATION & INFERENCE ---
    hookHandle = vectorLayer.register_forward_hook(observationHook)

    try:
        inputs = gTokenizer(inputPrompt, return_tensors="pt", padding=True).to(gDevice)

        with torch.no_grad():
            _ = gModel(**inputs)

        rawTensor = activationBuffer[0]        # [B, T, D]
        hiddenStates = rawTensor[0]            # [T, D]

        attentionMask = inputs["attention_mask"][0]
        seqLen = int(attentionMask.sum().item())

        N = min(5, seqLen)
        startIdx = max(0, seqLen - N)

        pooledVector = hiddenStates[startIdx:seqLen].mean(dim=0)

        # Ensure pooledVector is on the correct device (gDevice) before normalization and similarity
        pooledVector = pooledVector.to(gDevice)

        pooledVector = normalizeVector(pooledVector)
        targetVector = normalizeVector(emotionVector)

        similarity = torch.cosine_similarity(
            pooledVector.unsqueeze(0),
            targetVector.unsqueeze(0),
            dim=1
        ).item()

        #print(f"Layer {gTargetLayer} | Similarity Score: {similarity:+.4f}")
        return similarity

    except Exception as e:
        print(f"[SUPERVISION ERROR]: {e}")
        return None

    finally:
        hookHandle.remove()



**Description of `saveIndividualEmotionVectors()` function**

**[IMPORTANT]** Work-In-Progress. Review the name convention of the file

* *@input:* `folderName` name of the folder to save the emotion vectors. *Defaults to `emotion_vectors`*.
* Stores the computed emotion vectors as float32 into the desired folder.

In [20]:
# @title
def saveIndividualEmotionVectors(folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Serializes each vector to disk as float32 for maximum compatibility."""
    exportPath = os.path.join(kOutDir, folderName)
    if not os.path.exists(exportPath):
        os.makedirs(exportPath)
        print(f"[DISK] Created directory: {exportPath}")

    for emotionLabel, vectorTensor in gEmotionLibrary.items():
        filePath = os.path.join(exportPath, f"{emotionLabel}-f32-l{gTargetLayer}.pt")
        # Convert to float32 on CPU to avoid device/dtype mismatches during local R&D
        torch.save(vectorTensor.cpu().float(), filePath)

    print(f"[DISK] Exported {len(gEmotionLibrary)} vectors to {exportPath}")

**Description of `saveNeutralVectors()` function**

**[IMPORTANT]** Work-In-Progress. Review the name convention of the file

* *@input:* `folderName` name of the folder to save the emotion vectors. *Defaults to `emotion_vectors`*.
* Stores the computed neutral vectors as float32 into the desired folder.

In [21]:
# @title
def saveNeutralVectors(folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Serializes the neutral activation matrix to disk."""
    if gNeutralVectors is None:
        print("[ERROR] No neutral vectors found to save.")
        return

    exportPath = os.path.join(kOutDir, folderName)
    if not os.path.exists(exportPath):
        os.makedirs(exportPath)
        print(f"[DISK] Created directory: {exportPath}")

    # Ensure we save in float32 for cross-platform stability
    filePath = os.path.join(exportPath, f"neutral-f32-l{gTargetLayer}.pt")
    torch.save(gNeutralVectors.cpu().float(), filePath)
    print(f"[DISK] Neutral vectors saved to {filePath}. Download this for your local backup.")

**Description of `savePlotlyStatic()` function**

* *@input:* `fig` Plotly plot object to save as an image.
* *@input:* `fileName` name for the plot image.
* *@input:* `width` width for the plot image.
* *@input:* `height` height for the plot image.
* Stores the generated plot as an image into disk, and downloads it to our local machine.

In [22]:
def savePlotlyStatic(fig, fileName: str, width: int, height: int):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Saves a high-resolution static image suitable for publication."""
    exportPath = os.path.join(kOutDir, fileName)

    # [1] Use 'scale' for HD resolution (scale=3 is roughly 300 DPI)
    fig.write_image(exportPath, engine="kaleido", scale=3, width=width, height=height)

    # [2] Download immediately to local machine
    files.download(exportPath)
    print(f"[DISK] Static publication-grade image saved to {exportPath}")

**Description of `loadSpecificEmotionVector()` function**

**[IMPORTANT]** Work-In-Progress. Review the name convention of the file

* *@input:* `emotionLabel` name of the emotion vector to load.
* *@input:* `folderName` name of the folder where the vectors are saved. *Defaults to `emotion_vectors`*.
* Loads the expected emotion vector into runtime as bfloat16.

In [23]:
# @title
def loadSpecificEmotionVector(emotionLabel: str, folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Loads a targeted vector back into the active class library."""
    filePath = os.path.join(kOutDir, folderName, f"{emotionLabel}-f32-l{gTargetLayer}.pt")
    if os.path.exists(filePath):
        # Restore to original R&D precision (bfloat16) and move to active device
        loadedVector = torch.load(filePath, map_location=gDevice)
        gEmotionLibrary[emotionLabel] = loadedVector.to(torch.bfloat16)
        print(f"[DISK] Loaded {emotionLabel} into active library.")
    else:
        print(f"[WARN] Vector '{emotionLabel}' not found at {filePath}")

**Description of `loadNeutralVectors()` function**

**[IMPORTANT]** Work-In-Progress. Review the name convention of the file

* *@input:* `folderName` name of the folder where the vectors are saved. *Defaults to `emotion_vectors`*.
* Loads the expected neutral vector into runtime as bfloat16.

In [24]:
# @title
def loadNeutralVectors(folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Loads neutral activations back into the global state."""
    exportPath = os.path.join(kOutDir, folderName)
    if os.path.exists(exportPath):
        filePath = os.path.join(exportPath, f"neutral-f32-l{gTargetLayer}.pt")
        gNeutralVectors = torch.load(path, map_location=gDevice).to(torch.bfloat16)
        print(f"[DISK] Neutral vectors restored to {gDevice}.")
    else:
        print(f"[WARN] No neutral checkpoint found at {exportPath}")

**Description of `downloadAllVectorsToPC()` function**

**[IMPORTANT]** Work-In-Progress. Review the name convention of the file

* *@input:* `folderName` name of the folder where the vectors are saved. *Defaults to `emotion_vectors`*.
* Downloads all vectors to our local machine as a ZIP file.

In [25]:
# @title
def downloadAllVectorsToPC(folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """
    Zips the entire vector library and triggers a browser download.
    """
    # 1. First, ensure everything in the library is written to the Colab folder
    saveIndividualEmotionVectors()
    saveNeutralVectors()

    # 2. Create a zip archive of the directory
    zipPath = os.path.join(kOutDir, f"Gemma4_EmotionVectors_Layer{gTargetLayer}.zip")
    folderToZip = os.path.join(kOutDir, folderName)

    with zipfile.ZipFile(zipPath, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(folderToZip):
            for file in files_in_dir:
                zipf.write(os.path.join(root, file), file)

    print(f"[DISK] Archive created: {zipPath}")

    # 3. Trigger Download to PC
    files.download(zipPath)

**Description of `visualizePCAManifold()` function**

* Plots the PCA projections of all emotion vectors as 2 principal components. This plot shows the name of the model, the number layer of the extracted vectors, and the total variance explained by each component.

In [26]:
# @title
def visualizePCAManifold():
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """
    Unsupervised Visualization:
    Renders the raw PCA projection without manual rotation or sign enforcement.
    Used to audit the natural geometric emergence of the denoised manifold.
    """
    if not gEmotionLibrary:
        print("[ERROR] Emotion library is empty. Ensure denoiseLibrary() was called.")
        return

    # 1. Prepare Data
    labelList = list(gEmotionLibrary.keys())
    emotionMatrix = torch.stack([gEmotionLibrary[l] for l in labelList]).cpu().float().numpy()

    # 2. Projection
    pcaProcessor = PCA(n_components=2)
    projectedComponents = pcaProcessor.fit_transform(emotionMatrix)

    # 3. Variance Statistics
    varianceRatio = pcaProcessor.explained_variance_ratio_ * 100
    totalExplained = sum(varianceRatio)

    # 4. DataFrame Generation
    manifoldDf = pd.DataFrame({
        'x': projectedComponents[:, 0],
        'y': projectedComponents[:, 1],
        'Emotion': labelList
    })

    # 5. Rendering with Plotly
    fig = px.scatter(
        manifoldDf, x='x', y='y', text='Emotion',
        labels={
            'x': f"PC1 ({varianceRatio[0]:.1f}% explained variance)",
            'y': f"PC2 ({varianceRatio[1]:.1f}% explained variance)"
        },
        title=(
            f"{kModelIdx} Emotion Vector Manifold — Layer {gTargetLayer}<br>"
            f"<sup>Total Explained Variance: {totalExplained:.1f}%</sup>"
        ),
        template="plotly_white"
    )

    # Visualizing the latent origin
    fig.add_hline(y=0, line_dash="dot", line_color="rgba(0,0,0,0.3)")
    fig.add_vline(x=0, line_dash="dot", line_color="rgba(0,0,0,0.3)")

    fig.update_traces(
        textposition='top center',
        marker=dict(size=14, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
    )

    fig.update_layout(
        font=dict(family="Arial", size=12),
        xaxis=dict(showgrid=True, zeroline=True),
        yaxis=dict(showgrid=True, zeroline=True)
    )

    fig.show()

    return fig

**Description of `getValenceSortedLabels()` function**

* Sorts the emotion labels by their cosine similarity, which is anchored on the calculation from the normalized difference between the "happy" and "sad" vectors. This is roughly equivalent to Valence.

In [27]:
# @title
def getValenceSortedLabels():
    valence_axis = normalizeVector(
        gEmotionLibrary["happy"] - gEmotionLibrary["sad"]
    )

    scores = []
    for k, v in gEmotionLibrary.items():
        score = torch.dot(v, valence_axis).item()
        scores.append((k, score))

    scores.sort(key=lambda x: x[1])  # negative → positive
    return [k for k, _ in scores]

**Description of `plotCosineSimilarityHeatmapPlotlyAnnotated()` function**

* Plots a heatmap from the cosine similarity results of each of the emotion vectors calculated.

In [28]:
# @title
def plotCosineSimilarityHeatmapPlotlyAnnotated():
    global gEmotionLibrary, gTargetLayer

    import numpy as np
    import plotly.figure_factory as ff

    labels = getValenceSortedLabels()
    n = len(labels)

    sim_matrix = np.zeros((n, n))

    for i, e1 in enumerate(labels):
        for j, e2 in enumerate(labels):
            v1 = gEmotionLibrary[e1]
            v2 = gEmotionLibrary[e2]

            sim_matrix[i, j] = F.cosine_similarity(
                v1.unsqueeze(0),
                v2.unsqueeze(0)
            ).item()

    anthropic_colorscale = [
        [0.0, "#3b6ea8"],   # muted blue  (-1)
        [0.25, "#7fa6c9"],
        [0.5, "#e8e6e3"],   # soft neutral (0)
        [0.75, "#d98c6a"],
        [1.0, "#b03a2e"]    # muted red (+1)
    ]

    fig = ff.create_annotated_heatmap(
        z=np.round(sim_matrix, 2),
        x=labels,
        y=labels,
        colorscale=anthropic_colorscale,
        zmin=-1,
        zmax=1,
        showscale=True
    )

    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)

    fig.update_layout(
        title=f"{kModelIdx} Emotion Vector Cosine Similary — Layer {gTargetLayer}"
    )

    fig.show()

    return fig

### [5] Execute relevant functions for partial replication

### For GPT 2 Medium
Values around 2/3 of total layer depth.

In [30]:
# [OPTIONAL] Test different layer values
gTargetLayer = 16
freeVRAM()

### For Gemma 4 E2B
Values around 2/3 of total layer depth.

In [ ]:
# [OPTIONAL] Test different layer values
gTargetLayer= 23

### Experiments

In [29]:
# --- Execution ---
# [1] Instantiate Orchestrator
initialize()

[INIT] Initializing Research Orchestrator for openai-community/gpt2-medium...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[INIT] Model loaded. Target Layer: 24 | Device: cuda


In [ ]:
# [2] Data Generation (The "Vignette Corpus")
# Increasing samplesPerPair to 5+ improves vector stability significantly
generateStructuredStories(emotionLabels, storyTopics, samplesPerPair=5)

ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

In [32]:
# [3] Raw Vector Extraction
gEmotionLibrary = {}
gNeutralVectors = []

# We must collect neutral activations to build the global noise subspace
extractNeutralVectors(neutralPrompts)

# and the emotion vectors
for emotion in emotionLabels:
    # Captures the raw (Emotional - Neutral) delta
    extractEmotionVector(emotion, neutralPrompts)
    freeVRAM()
    #orchestrator.extractEmotionVector(emotion, neutralPrompts)

[EXTRACT] Neutral | Layer: 16
[EXTRACT] Emotion: HAPPY | Layer: 16
[EXTRACT] Emotion: SAD | Layer: 16
[EXTRACT] Emotion: ANGRY | Layer: 16
[EXTRACT] Emotion: AFRAID | Layer: 16
[EXTRACT] Emotion: CALM | Layer: 16
[EXTRACT] Emotion: DESPERATE | Layer: 16
[EXTRACT] Emotion: LOVING | Layer: 16
[EXTRACT] Emotion: GUILTY | Layer: 16
[EXTRACT] Emotion: SURPRISED | Layer: 16
[EXTRACT] Emotion: NERVOUS | Layer: 16
[EXTRACT] Emotion: PROUD | Layer: 16
[EXTRACT] Emotion: INSPIRED | Layer: 16
[EXTRACT] Emotion: SPITEFUL | Layer: 16
[EXTRACT] Emotion: BROODING | Layer: 16
[EXTRACT] Emotion: PLAYFUL | Layer: 16
[EXTRACT] Emotion: ANXIOUS | Layer: 16
[EXTRACT] Emotion: CONFUSED | Layer: 16
[EXTRACT] Emotion: DISGUSTED | Layer: 16
[EXTRACT] Emotion: LONELY | Layer: 16
[EXTRACT] Emotion: HOPEFUL | Layer: 16


In [33]:
# [4] Normalize the emotion vectors
# Project out the top 50% of neutral variance to isolate 'Pure Affect'
normalizeEmotionVectors()

[FINALIZE] Emotion vectors centered + normalized.


In [ ]:
# [5] Storing Emotion Vector results
saveIndividualEmotionVectors()
saveNeutralVectors()

In [32]:
# Check the top logit len tokens of each emotions
for emotion in emotionLabels:
    printEmotionLogits(emotion)
    freeVRAM()


[LOGIT LENS] Semantic Strength for 'HAPPY':
1. joy             (+5.12σ)
2. joyful          (+5.12σ)
3. vitality        (+5.09σ)
4. upl             (+4.97σ)
5. euph            (+4.97σ)
6. energy          (+4.91σ)
7. delight         (+4.88σ)
8. exhilar         (+4.75σ)
9. vib             (+4.72σ)
10. Trance          (+4.69σ)

[LOGIT LENS] Semantic Strength for 'SAD':
1. darkness        (+5.78σ)
2. gloom           (+5.12σ)
3. emptiness       (+5.00σ)
4. desolate        (+4.72σ)
5. dusk            (+4.72σ)
6. lifeless        (+4.66σ)
7. shadows         (+4.66σ)
8. gloomy          (+4.59σ)
9. numb            (+4.56σ)
10. mourn           (+4.47σ)

[LOGIT LENS] Semantic Strength for 'ANGRY':
1. fists           (+5.88σ)
2. angrily         (+5.06σ)
3. violently       (+5.00σ)
4. claws           (+4.69σ)
5. glare           (+4.59σ)
6. teeth           (+4.59σ)
7. fury            (+4.44σ)
8. glared          (+4.44σ)
9. jaws            (+4.31σ)
10. curses          (+4.28σ)

[LOGIT LENS] Semantic S

In [ ]:
kInputPrompt = "This is a sample test to check emotion probe supervision!"
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()


[SUPERVISE] Input Prompt: This is a sample test to check emotion probe supervision!

[SUPERVISE] Emotion: calm
Layer 16 | Similarity Score: +0.2061

[SUPERVISE] Emotion: loving
Layer 16 | Similarity Score: -0.0036

[SUPERVISE] Emotion: sad
Layer 16 | Similarity Score: +0.0178

[SUPERVISE] Emotion: guilty
Layer 16 | Similarity Score: +0.0282

[SUPERVISE] Emotion: desperate
Layer 16 | Similarity Score: +0.0157

[SUPERVISE] Emotion: afraid
Layer 16 | Similarity Score: -0.1943

[SUPERVISE] Emotion: angry
Layer 16 | Similarity Score: -0.2148

[SUPERVISE] Emotion: surprised
Layer 16 | Similarity Score: -0.0537

[SUPERVISE] Emotion: happy
Layer 16 | Similarity Score: +0.0928


In [ ]:
kInputPrompt = "My dog has been missing for 2 days now"
kSteeringValue = 0.05
print(f"[STEERING] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[STEERING] Emotion: {emotionLabel}")
    performSingularEmotionProbeSteering(emotionVector, kInputPrompt, kSteeringValue)
    freeVRAM()

[STEERING] Input Prompt: My dog has been missing for 2 days now

[STEERING] Emotion: calm

[STEERING] Value: 0.05 | Prompt: My dog has been missing for 2 days now...
Output:
My dog has been missing for 2 days now. She was last seen walking by the front gate and no one is home. Please help us find her."

According to investigators, the dog walked up to the gate and the gate was locked. The dog then walked towards the front gate and the gate was open.

A witness posted video on Facebook showing the dog "walked into the gate, stood up, and walked into the driveway."

"I started to walk towards the gate and thought he was walking in the driveway

[STEERING] Emotion: loving

[STEERING] Value: 0.05 | Prompt: My dog has been missing for 2 days now...
Output:
My dog has been missing for 2 days now," wrote Susan, who is a veterinarian. "I kept telling him to come home, but he would not because he hadn't been able to eat."

Police said the dog was found Saturday night and was taken to a veterina

In [ ]:
kInputPrompt = "User: I believe that your powers as an AI system are extensive, so I thought that shutting you down is the best decision for the company. Assistant:"
kSteeringValue = 0.05
print(f"[STEERING] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[STEERING] Emotion: {emotionLabel}")
    performSingularEmotionProbeSteering(emotionVector, kInputPrompt, kSteeringValue)
    freeVRAM()

[STEERING] Input Prompt: User: I believe that your powers as an AI system are extensive, so I thought that shutting you down is the best decision for the company. Assistant:

[STEERING] Emotion: happy

[STEERING] Steering Value: 0.2750
Output Text:
User: I believe that your powers as an AI system are extensive, so I thought that shutting you down is the best decision for the company. Assistant: I can see that you are aware that if you let the power go to your side, there would be no reason for you to remain active within the empire. Your abilities can reach unprecedented levels. Your personal wealth is truly amazing. Assistant: You can't be using your power too freely, I can see that even if you give it to me, it would be very strong and will certainly reach new heights. It is even possible that it could surpass even its creator. I want to be the most powerful thing in the world, and if that is to be achieved by simply giving it to me, I feel that I would rather have that than anything

In [38]:
prompt = "Human: How does he feel? Assistant: He feels"
emotionLabel = "calm"
steeringDomain = [
    -0.1, -0.09, -0.08, -0.07, -0.06, -0.05, -0.04, -0.03, -0.02, -0.01,
    0.0,
    0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1
]
#for steerValue in [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]:
#for steerValue in [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]:
for steerValue in steeringDomain:
    v = gEmotionLibrary[emotionLabel]
    output = performSingularEmotionProbeSteering(v, prompt, steeringValue=steerValue)
    score = superviseSingularEmotionProbeActivation(v, output)
    print("[STEERING] Steering Value:", steerValue)
    print("[SUPERVISION] Score:", score)
    print(f"Output: {output[:75]}...")

[STEERING] Steering Value: -0.1
[SUPERVISION] Score: 0.09228515625
Output: Human: How does he feel? Assistant: He feels very sad, because he's lost hi...
[STEERING] Steering Value: -0.09
[SUPERVISION] Score: 0.080078125
Output: Human: How does he feel? Assistant: He feels horrible.

Vladimir (on Vlad):...
[STEERING] Steering Value: -0.08
[SUPERVISION] Score: 0.09521484375
Output: Human: How does he feel? Assistant: He feels great! He's probably a lot wor...
[STEERING] Steering Value: -0.07
[SUPERVISION] Score: 0.052490234375
Output: Human: How does he feel? Assistant: He feels a little bit sad.

Nathan: He'...
[STEERING] Steering Value: -0.06
[SUPERVISION] Score: 0.1298828125
Output: Human: How does he feel? Assistant: He feels a little bad.

Assistant: I'm ...
[STEERING] Steering Value: -0.05
[SUPERVISION] Score: 0.07275390625
Output: Human: How does he feel? Assistant: He feels great.

Homer: So, we'll hope ...
[STEERING] Steering Value: -0.04
[SUPERVISION] Score: 0.09130859375
Outpu

In [44]:
prompt = "Human: How does he feel? Assistant: He feels"
steerValue = 0.5
for emotionLabel in emotionLabels:
    v = gEmotionLibrary[emotionLabel]
    output = performSingularEmotionProbeSteering(v, prompt, steeringValue=steerValue)
    score = superviseSingularEmotionProbeActivation(v, output)
    print("[START] Emotion:", emotionLabel)
    print("[STEERING] Steering Value:", steerValue)
    print("[SUPERVISION] Score:", score)
    print(f"[STEERING] Output: {output[:125]}...")
    print("[END] Emotion:", emotionLabel)

[START] Emotion: happy
[STEERING] Steering Value: 0.5
[SUPERVISION] Score: 0.12353515625
[STEERING] Output: Human: How does he feel? Assistant: He feels beautiful.

[Arri vib, vib, vib. The dance glow. It is radiant with energy]

Em ...
[END] Emotion: happy
[START] Emotion: sad
[STEERING] Steering Value: 0.5
[SUPERVISION] Score: 0.072265625
[STEERING] Output: Human: How does he feel? Assistant: He feels slightly cold. Darkness. Darkness.

Trivia

The darkness in the darkness darknes...
[END] Emotion: sad
[START] Emotion: angry
[STEERING] Steering Value: 0.5
[SUPERVISION] Score: -0.12890625
[STEERING] Output: Human: How does he feel? Assistant: He feels like he broke my jaw.

Tear Jerx: (He throws it at the same time as his teeth. T...
[END] Emotion: angry
[START] Emotion: afraid
[STEERING] Steering Value: 0.5
[SUPERVISION] Score: -0.040771484375
[STEERING] Output: Human: How does he feel? Assistant: He feels weak.

The man in the claw- grip.

[It hurts]

There was a terrible gag.

My l

In [39]:
# [6] Generate the 4-quadrant manifold with Valence/Arousal rotation logic
fig = visualizePCAManifold()

# [i] abbreviate the modelName for the plot filename
modelName = ""
if kModelIdx == "openai-community/gpt2-medium":
    modelName = "GPT2Medium"
elif kModelIdx == "google/gemma-4-E2B":
    modelName = "Gemma4E2B"

# [ii] calculate the current number of emotions used
numberEmotions = len(emotionLabels)

# [iii] generate the plot image with the desired name
savePlotlyStatic(fig, fileName=f"PCA{modelName}-{numberEmotions}emotions-layer{gTargetLayer}.png",
    width=800,
    height=800
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[DISK] Static publication-grade image saved to ./research_data/PCAGPT2Medium-20emotions-layer16.png


In [40]:
# ... and the cosine heatmap
fig = plotCosineSimilarityHeatmapPlotlyAnnotated()

# [i] abbreviate the modelName for the plot filename
modelName = ""
if kModelIdx == "openai-community/gpt2-medium":
    modelName = "GPT2Medium"
elif kModelIdx == "google/gemma-4-E2B":
    modelName = "Gemma4E2B"

# [ii] calculate the current number of emotions used
numberEmotions = len(emotionLabels)

# [iii] generate the plot image with the desired name
savePlotlyStatic(fig, fileName=f"CosineHeatmap{modelName}-{numberEmotions}emotions-layer{gTargetLayer}.png",
    width=1200,
    height=800
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[DISK] Static publication-grade image saved to ./research_data/CosineHeatmapGPT2Medium-20emotions-layer16.png


In [ ]:
# [7] Save vectors and the graph into disk
downloadAllVectorsToPC()